### 规格化欧洲动物数据

In [40]:
import pandas as pd
# 左连接函数
def merge_left(data_left,data_right,item,year,on="OBJECTID",label="animals_label"):
    data_item = data_right.loc[data_right[label]==item].reset_index(drop=True)
    data_left = data_left.merge(data_item[[year,on]],on=on,how="left")
    return data_left.rename(columns={year:item}) # 改名

# 列相加函数
def sum_cols(data,cols_to_convert,new_col_name):
    def safe_numeric(x):
        try:
            return pd.to_numeric(x)
        except ValueError:
            return 0

    for col in cols_to_convert:
        if data[col].dtype == 'str':
            data[col] = data[col].str.replace(',', '')  # remove commas
        elif data[col].dtype == 'float64':
            continue
        else:
            print('Warning: column {} is not a string and float64,but is {}'.format(col,data[col].dtype))
            
    data[new_col_name] = data[cols_to_convert].applymap(safe_numeric).sum(axis=1) # 新的一列

    # 删除这些列
    return data.drop(columns=cols_to_convert)

In [37]:
# 批量处理
for y in range(1977,2024):
    livestock = pd.read_csv("eurostat_animal_livestocks/"+str(y)+".csv")

    # 初始化animal
    animal = livestock.drop_duplicates(subset=["geo\TIME_PERIOD"]).loc[:,["geo\TIME_PERIOD","geo_label","unit_label"]].reset_index(drop=True)
    animal["year"] = y

    # 处理奶牛
    item = "Dairy cows"
    animal = merge_left(animal,livestock,item,str(y),on="geo\TIME_PERIOD")

    # 处理肉牛
    item = ["Heifers, 1 year old, for slaughter"]
    animal = merge_left(animal,livestock,item[-1],str(y),on="geo\TIME_PERIOD")

    item.append("Heifers, 2 years old or over, for slaughter")
    animal = merge_left(animal,livestock,item[-1],str(y),on="geo\TIME_PERIOD")

    item.append("Bovine animals, less than 1 year old, for slaughter")
    animal = merge_left(animal,livestock,item[-1],str(y),on="geo\TIME_PERIOD")

    # 相加合并这三列形成肉牛总量
    animal = sum_cols(animal,item,"Beef cattle")

    # 处理其它牛，live bovine animal 减去奶牛和肉牛
    item = "Live bovine animals"
    animal = merge_left(animal,livestock,item,str(y),on="geo\TIME_PERIOD")
    if animal["Live bovine animals"].dtype == 'float64' and animal["Dairy cows"].dtype == 'float64' and animal["Beef cattle"].dtype == 'float64':
        animal["excl cattle"] = animal["Live bovine animals"] - animal["Dairy cows"] - animal["Beef cattle"]
    else:
        print("其它牛计算失败，因为数据类型不对")
    animal.drop(columns="Live bovine animals",inplace=True)

    # 处理猪
    item = "Fattening pigs, live weight 50 kg or over"
    animal = merge_left(animal,livestock,item,str(y),on="geo\TIME_PERIOD")

    # 处理羊
    item = ["Live sheep"]
    animal = merge_left(animal,livestock,item[-1],str(y),on="geo\TIME_PERIOD")

    item.append("Live goats")
    animal = merge_left(animal,livestock,item[-1],str(y),on="geo\TIME_PERIOD")

    # 相加合并这2列形成羊总量
    animal = sum_cols(animal,item,"Sheep")

    # 保存
    # animal.to_csv("animals_差禽类/"+str(y)+".csv",index=False,encoding="utf-8-sig")


In [28]:
# 地区标准化排列，字符串长度为2的就是国家，长度为3的就是州，长度为4的就是县
for y in range(1977,2024):
    animal = pd.read_csv("D:/中科院数据下载/废弃数据/animals_差禽类/"+str(y)+".csv")
    # 提取国家、州、县形成字典
    geo_dict = animal.set_index("geo\TIME_PERIOD")["geo_label"].to_dict()

    # 提取县
    # animal_county = animal[animal["geo\TIME_PERIOD"].str.contains("\d{2}$",regex=True)]
    animal_county = animal[animal["geo\TIME_PERIOD"].str.len() == 4]
    animal_county = animal_county.rename(columns={"geo\TIME_PERIOD":"county","geo_label":"county_label"})

    # 补全县前面的州
    animal_county.loc[:, 'state'] = animal_county['county'].str[:-1]
    animal_county.loc[:, 'state_label'] = animal_county['state'].map(geo_dict)

    # 将新列移动到最左边
    cols = animal_county.columns.tolist()
    cols.insert(0, cols.pop(cols.index('state_label')))
    cols.insert(0, cols.pop(cols.index('state')))
    animal_county = animal_county.reindex(columns=cols)

    # 补全县前面的国家
    animal_county.loc[:, 'country'] = animal_county['state'].str[:-1]
    animal_county.loc[:, 'country_label'] = animal_county['country'].map(geo_dict)

    # 将新列移动到最左边
    cols = animal_county.columns.tolist()
    cols.insert(0, cols.pop(cols.index('country_label')))
    cols.insert(0, cols.pop(cols.index('country')))
    animal_county = animal_county.reindex(columns=cols)

    # 提取州
    # animal_state = animal[animal["geo\TIME_PERIOD"].str.contains("^[^\d]*\d{1}$", regex=True)]
    animal_state = animal[animal["geo\TIME_PERIOD"].str.len() == 3]
    animal_state = animal_state.rename(columns={"geo\TIME_PERIOD":"state","geo_label":"state_label"})

    # 补全州前面的国家
    animal_state.loc[:, 'country'] = animal_state['state'].str[:-1]
    animal_state.loc[:, 'country_label'] = animal_state['country'].map(geo_dict)

    # 将新列移动到最左边
    cols = animal_state.columns.tolist()
    cols.insert(0, cols.pop(cols.index('country_label')))
    cols.insert(0, cols.pop(cols.index('country')))
    animal_state = animal_state.reindex(columns=cols)

    # 用空值补全县的列
    animal_state.insert(4,"county","")
    animal_state.insert(5,"county_label","")

    # 提取国家
    # animal_country = animal[animal["geo\TIME_PERIOD"].str.contains(".*\D$", regex=True)]
    animal_country = animal[animal["geo\TIME_PERIOD"].str.len() == 2]
    animal_country = animal_country.rename(columns={"geo\TIME_PERIOD":"country","geo_label":"country_label"})

    animal_country.insert(2,"state","")
    animal_country.insert(3,"state_label","")
    animal_country.insert(4,"county","")
    animal_country.insert(5,"county_label","")

    # 合并
    animal = pd.concat([animal_county,animal_state,animal_country],axis=0,ignore_index=True)

    # 保存
    animal.to_csv("D:/中科院数据下载/废弃数据/animal_geo_ok_差禽/"+str(y)+".csv",index=False,encoding="utf-8-sig")

    

In [29]:
# 欧盟禽类数据补全
# 需要的数据有：“之前已经处理好的动物”、“禽类总量数据poultry_annual_data”、“account数据”
# 步骤先用xx年的poultry_annual_data数据计算肉蛋鸡总量，再用account计算计算比例，最后得到各县肉蛋鸡数量，
# 然后merge与已经处理好的数据合并得到最终结果
import pandas as pd
import os 
import re

def is_match(pattern, string):
    return bool(re.search(pattern, string))



In [30]:
# 一年一年来
year_begin = 1980
year_end = 2021
for y in range(year_begin,year_end+1):
     # Load the poultry_annual_data dataset
    base_path = 'D:/中科院数据下载/废弃数据/animal_geo_ok_差禽/poultry_annual_data'
    filename = str(y) + ".csv"
    poultry_data_path = os.path.join(base_path, filename)
    poultry_data = pd.read_csv(poultry_data_path)

    # 计算各个国家肉蛋鸡总量
    # Filter relevant data for calculations
    relevant_poultry_data = poultry_data[
    (poultry_data['hatchitm_label'].isin(['Chicks hatched', 'Chicks placed'])) & 
    (poultry_data['animals_label'].isin(['Chicks of laying hen breeds (laying)', 'Chicks of meat broiler breeds (fattening)']))
    ]

    # Group by country and animals_label, then sum up the values
    poultry_totals = relevant_poultry_data.groupby(['geo\\TIME_PERIOD', 'animals_label'])[str(y)].sum().reset_index()

    # Pivot the table for easier calculations
    poultry_totals_pivot = poultry_totals.pivot(index='geo\\TIME_PERIOD', columns='animals_label', values=str(y)).reset_index()
    poultry_totals_pivot.columns.name = None  # Remove the pivot column name for clarity

    # Rename the columns for clarity
    poultry_totals_pivot.rename(columns={
    'Chicks of laying hen breeds (laying)': 'Total Layers',
    'Chicks of meat broiler breeds (fattening)': 'Total Broilers'
    }, inplace=True)

    # # Load the anaimal_Economic_accounts_for_agriculture dataset
    base_path = 'D:/中科院数据下载/eurostat/anaimal_Economic_accounts_for_agriculture'
    filename = str(y) + ".csv"
    economic_data_path = os.path.join(base_path, filename)
    economic_data = pd.read_csv(economic_data_path)

    # 计算蛋鸡
    # Filter relevant data for egg production values
    relevant_economic_data = economic_data[
        (economic_data['indic_ag_label'] == "Production value at basic price") &
        (economic_data['itm_newa_label'] == "Eggs") &
        (economic_data['unit_label'] == "Million euro")
    ].copy()

    # Group by country/county and sum up the values
    egg_production_totals = relevant_economic_data

    # Create a new column to determine whether the row is a country, state, or county
    egg_production_totals.loc[:, 'Type'] = egg_production_totals['geo\\TIME_PERIOD'].apply(
        lambda x: 'County' if is_match("\d{2}$",x) else ('State' if is_match("^[^\d]*\d{1}$",x) else 'Country')
    )

    # Split the dataframe into separate ones for countries and counties
    country_egg_totals = egg_production_totals[egg_production_totals['Type'] == 'Country']
    county_egg_totals = egg_production_totals[egg_production_totals['Type'] == 'County']

    # Merge the county and country dataframes to calculate each county's share of national egg production
    county_egg_share = county_egg_totals.merge(country_egg_totals, left_on=county_egg_totals['geo\\TIME_PERIOD'].str[:-2], right_on='geo\\TIME_PERIOD', suffixes=('_county', '_country'))
    county_egg_share['Egg_Share'] = county_egg_share[str(y)+'_county'] / county_egg_share[str(y)+'_country']

    # Keep only the necessary columns
    county_egg_share = county_egg_share[['geo\\TIME_PERIOD_county', 'Egg_Share']]
    county_egg_share.rename(columns={'geo\\TIME_PERIOD_county': 'geo\\TIME_PERIOD'}, inplace=True)

    # Correcting the merge to ensure the column names align
    county_layer_data = county_egg_share.merge(
        poultry_totals_pivot[['geo\\TIME_PERIOD', 'Total Layers']], 
        left_on=county_egg_share['geo\\TIME_PERIOD'].str[:-2], 
        right_on='geo\\TIME_PERIOD',
        how='left',
        suffixes=('_county', '_country')
    )

    # Calculate each county's estimated number of layer chickens
    county_layer_data['County Layers'] = county_layer_data['Egg_Share'] * county_layer_data['Total Layers']

    # Keep only the necessary columns
    county_layer_data_final = county_layer_data[['geo\\TIME_PERIOD_county', 'County Layers']].copy()
    county_layer_data_final.rename(columns={'geo\\TIME_PERIOD_county': 'County Code'}, inplace=True)


    # 接下来是肉鸡，用的account里面的poultry比例
    # Filter relevant data for egg production values
    relevant_economic_data = economic_data[
        (economic_data['indic_ag_label'] == "Production value at basic price") &
        (economic_data['itm_newa_label'] == "Poultry") &
        (economic_data['unit_label'] == "Million euro")
    ].copy()

    # Group by country/county and sum up the values
    poultry_production_totals = relevant_economic_data

    # Create a new column to determine whether the row is a country, state, or county
    poultry_production_totals.loc[:, 'Type'] = poultry_production_totals['geo\\TIME_PERIOD'].apply(
        lambda x: 'County' if is_match("\d{2}$",x) else ('State' if is_match("^[^\d]*\d{1}$",x) else 'Country')
    )

    # Split the dataframe into separate ones for countries and counties
    country_poultry_totals = poultry_production_totals[poultry_production_totals['Type'] == 'Country']
    county_poultry_totals = poultry_production_totals[poultry_production_totals['Type'] == 'County']

    # Merge the county and country dataframes to calculate each county's share of national egg production
    county_poultry_share = county_poultry_totals.merge(country_poultry_totals, left_on=county_poultry_totals['geo\\TIME_PERIOD'].str[:-2], right_on='geo\\TIME_PERIOD', suffixes=('_county', '_country'))
    county_poultry_share['poultry_Share'] = county_poultry_share[str(y)+'_county'] / county_poultry_share[str(y)+'_country']

    # Keep only the necessary columns
    county_poultry_share = county_poultry_share[['geo\\TIME_PERIOD_county', 'poultry_Share']]
    county_poultry_share.rename(columns={'geo\\TIME_PERIOD_county': 'geo\\TIME_PERIOD'}, inplace=True)


    # Correcting the merge to ensure the column names align
    county_broiler_data = county_poultry_share.merge(
        poultry_totals_pivot[['geo\\TIME_PERIOD', 'Total Broilers']], 
        left_on=county_poultry_share['geo\\TIME_PERIOD'].str[:-2], 
        right_on='geo\\TIME_PERIOD',
        how='left',
        suffixes=('_county', '_country')
    )

    # Calculate each county's estimated number of layer chickens
    county_broiler_data['County broiler'] = county_broiler_data['poultry_Share'] * county_broiler_data['Total Broilers']

    # Keep only the necessary columns
    county_broiler_data_final = county_broiler_data[['geo\\TIME_PERIOD_county', 'County broiler']].copy()
    county_broiler_data_final.rename(columns={'geo\\TIME_PERIOD_county': 'County Code'}, inplace=True)

    # 与之前搞好的动物数据进行合并
    base_path = 'D:/中科院数据下载/废弃数据/animal_geo_ok_差禽'
    filename = str(y) + ".csv"
    animal_ok_path = os.path.join(base_path, filename)
    animal_ok_data = pd.read_csv(animal_ok_path)

    animal_ok_data = animal_ok_data.merge(county_layer_data_final,left_on="county",right_on="County Code",how="left")

    animal_ok_data = animal_ok_data.merge(county_broiler_data_final,left_on="county",right_on="County Code",how="left")

    animal_ok_data.drop(columns=["County Code_x","County Code_y"],inplace=True)

    # 保存
    animal_ok_data.to_csv("D:/中科院数据下载/eurostat/animal_geo_ok/"+str(y)+'.csv',index=False,encoding="utf-8-sig")

ds


In [17]:
# 与FAO数据校对
# 欧盟多个国家的校对
def single_proofread(data,data_fao,data_country_colname="country",data_fao_item_colname="item"):
    # data是之前已经整理好的数据，现在要被校对
    # data_fao是fao上的国家总量
    # data_country是data里面的国家列名
    # data_fao_item是fao里面的动物种类列名
    Year = data['year'].iloc[3]
    count_zero = 0
    count_error = 0
    data['标记'] = ""

    # 首先筛选出动物种类
    animal_species = data_fao[data_fao_item_colname].unique() 
    value_name = "Value" if "Value" in data_fao.columns else "value"
    data_fao[value_name] = data_fao[value_name].astype(str)
    data_fao[value_name] = data_fao[value_name].str.replace(',', '')
    data_fao[value_name] = pd.to_numeric(data_fao[value_name], errors='coerce')

    # 筛选出国家
    country_species = data[data_country_colname].unique()

    data_only_state = data[data['county'].isna() & data['state'].notna()] # 取出州的
    # 仅保留县
    data.dropna(subset=['county','state'],inplace=True)
    r = data.iloc[:,0:6] # 存储比例
    r['year'] = Year
    # data = data[data['county'].notna() & data['state'].notna()] # 仅保留县

    for animal in animal_species:
        
        # 首先判断这个动物种类是否在data里面
        if animal in data.columns:
            r[animal] = ''
            # 首先转化列里面数值保证能进行四则运算
            r[animal] = r[animal].astype(str)
            r[animal] = r[animal].str.replace(',', '')  # remove commas
            r[animal] = pd.to_numeric(r[animal], errors='coerce').fillna(0)

            # 首先转化列里面数值保证能进行四则运算
            data[animal] = data[animal].astype(str)
            data[animal] = data[animal].str.replace(',', '')  # remove commas
            data[animal] = pd.to_numeric(data[animal], errors='coerce')

            for country in country_species:
                # 获取FAO数据
                fao_value = data_fao[(data_fao[data_fao_item_colname] == animal) & (data_fao['Area'] == country)][value_name]
                if not fao_value.empty:
                    if pd.to_numeric(fao_value.iloc[0], errors='coerce') >= 0:
                        fao_value = fao_value.iloc[0]

                        # 计算国家内各县的总量(注意不要加重复了)
                        country_total = data_only_state[data_only_state[data_country_colname] == country][animal].sum() # 总量应该从州取
                        if country_total==0:
                            # 如果总量等于0，则从县取
                            country_total = data[data[data_country_colname] == country][animal].sum() # 总量应该从县取

                        if abs(fao_value-country_total) > 0.2*max(fao_value,country_total):
                            # 标记一下这一行，然后跳过,区分是有数据还是没有数据导致的
                            count_error += 1
                            if country_total==0 and fao_value != 0:
                                # print("存在缺失")
                                count_zero += 1
                                # 用fao_value代替country_total,country_total为0的化，国家里面的县也会为零，那这些县按照历史比例分配下去
                                # data.loc[data[data_country_colname] == country, animal] = r.loc[r[data_country_colname]==country , animal] * fao_value

                            data.loc[data[data_country_colname] == country, "标记"] += "种类："+animal+","+"FAO总量："+str(fao_value)+","+"国家总量："+str(country_total)+"; "
                            # data.loc[data[data_country] == country, "标记"] += "种类："+animal+","
                            # print("国家为{}，种类为{}的fao总量为{}，对应国家数据总量{}".format(country,animal,fao_value,country_total))
                            
                        # 如果国家总量大于0，则根据占比分配FAO数据
                        if country_total > 0:
                            r.loc[r[data_country_colname]==country,animal] = (data[animal] / country_total) 
                            data.loc[data[data_country_colname] == country, animal] = (data[animal] / country_total) * fao_value
    print("缺失占比为{}".format(count_zero/count_error))
    return data,r


In [57]:
# 规范化fao数据，注意单位
# 输入所有年份的fao原始数据,输出分年份规范化好的fao数据
def fao_standard(fao_data,params,target_path):
    # params是字典，键是fao数据的动物种类名称，值是键对应的要转化的种类名称，所有值是列表则证明要将这两项相加
    # 确保数值列可以做四则运算
    value_name = "Value" if "Value" in fao_data.columns else "value"
    fao_data[value_name] = fao_data[value_name].astype(str)
    fao_data[value_name] = fao_data[value_name].str.replace(',', '')
    fao_data[value_name] = pd.to_numeric(fao_data[value_name], errors='coerce')

    # 单位转化
    fao_data.loc[fao_data['Unit'] == 'An', 'Value'] /= 1000
    fao_data.loc[fao_data['Unit'] == 'An', 'Unit'] = '1000An'

    for year,year_group in fao_data.groupby("Year"):
        group = pd.DataFrame()
        # 一年一年,在此基础上再一个一个国家来
        for country,country_group in year_group.groupby("Area"):

            # fao_data数据只有Item列需要,用replace替换
            for item in params:

                if isinstance(item,str):

                    # 首先确保item在列的值里面
                    if item in list(country_group["Item"]):
                        country_group["Item"] = country_group['Item'].replace(item,params[item])

                elif isinstance(item,tuple):

                    # 先相加，按照年份，然后形成新的一个值
                    for i in range(1,len(item)):
                        if (item[i] in list(country_group["Item"])) and (item[0] in list(country_group["Item"])) :
                            country_group.loc[country_group["Item"]==item[0],value_name] += country_group[country_group['Item']==item[i]][value_name].values[0]
                    country_group["Item"] = country_group['Item'].replace(item[0],params[item]) 

                else:
                    print("params输入格式错误,错误的键为{}".format(item))
            # 国家合并在一起
            group = pd.concat([group,country_group],axis=0)

        # 保存
        group.to_csv(target_path+str(year)+".csv")

In [58]:
params = {
    ("Raw milk of cattle"):"Dairy cows",
    ("Meat of cattle with the bone, fresh or chilled","Meat of buffalo, fresh or chilled"):"Beef cattle",
    # "excl cattle":
    "Meat of pig with the bone, fresh or chilled":"Fattening pigs, live weight 50 kg or over",
    "Sheep and Goats":"Sheep",
    "Hen eggs in shell, fresh":"County Layers",
    "Meat of chickens, fresh or chilled":"County broiler"
}


In [59]:
import pandas as pd
# 规范化fao数据
target_path = 'D:/中科院数据下载/eurostat/animal_geo_ok/FAO数据/'
fao_data = pd.read_csv('D:/中科院数据下载/eurostat/animal_geo_ok/FAO数据/animal_fao.csv')
fao_standard(fao_data,params,target_path)

In [21]:
import pandas as pd
# data = pd.read_csv("D:/中科院数据下载/eurostat/animal_geo_ok/"+str(y)+".csv")
# data.dropna(subset=['county','state'],inplace=True)
# r = data.iloc[:,0:6] # 存储比例
r = pd.DataFrame()

# 进行校对
# 与FAO数据校对
year_begin = 1980
year_end = 2021
for y in range(year_begin,year_end+1):
    data = pd.read_csv("D:/中科院数据下载/eurostat/animal_geo_ok/"+str(y)+".csv")
    data_fao = pd.read_csv("D:/中科院数据下载/eurostat/animal_geo_ok/FAO数据/"+str(y)+".csv")
    data_ok,r_tmp = single_proofread(data,data_fao,data_country_colname="country_label",data_fao_item_colname="Item")

    r = pd.concat([r,r_tmp],axis=0)
    # data_ok.to_csv("动物_fao_ok/"+str(y)+".csv",index=False,encoding="utf-8-sig")

缺失占比为0.8571428571428571
缺失占比为0.8586956521739131
缺失占比为0.8478260869565217
缺失占比为0.8369565217391305
缺失占比为0.8260869565217391
缺失占比为0.8131868131868132
缺失占比为0.8241758241758241
缺失占比为0.7666666666666667
缺失占比为0.7444444444444445
缺失占比为0.7222222222222222
缺失占比为0.6781609195402298
缺失占比为0.6704545454545454
缺失占比为0.7297297297297297
缺失占比为0.7222222222222222
缺失占比为0.7222222222222222
缺失占比为0.6410256410256411
缺失占比为0.625
缺失占比为0.6016949152542372
缺失占比为0.5714285714285714
缺失占比为0.5727272727272728
缺失占比为0.5042016806722689
缺失占比为0.45614035087719296
缺失占比为0.44545454545454544
缺失占比为0.4537037037037037
缺失占比为0.5045045045045045
缺失占比为0.656
缺失占比为0.7388059701492538
缺失占比为0.330188679245283
缺失占比为0.39473684210526316
缺失占比为0.3925233644859813
缺失占比为0.30275229357798167
缺失占比为0.2897196261682243
缺失占比为0.3425925925925926
缺失占比为0.3611111111111111
缺失占比为0.37962962962962965
缺失占比为0.36538461538461536
缺失占比为0.3867924528301887
缺失占比为0.38317757009345793
缺失占比为0.3434343434343434
缺失占比为0.3191489361702128
缺失占比为0.32608695652173914
缺失占比为0.23809523809523808


In [22]:
r.to_csv("D:/中科院数据下载/eurostat/animal_geo_ok/比例县.csv",index=False,encoding='utf-8-sig')

In [33]:
# 与FAO数据校对
# 欧盟多个国家的校对
def single_proofread_r(data,data_fao,r,data_country_colname="country",data_fao_item_colname="item"):
    # data是之前已经整理好的数据，现在要被校对
    # data_fao是fao上的国家总量
    # data_country是data里面的国家列名
    # data_fao_item是fao里面的动物种类列名
    # r是所有的历史比例
    Year = data['year'].iloc[3]
    count_zero = 0
    count_error = 0
    data['标记'] = ""

    # 首先筛选出动物种类
    animal_species = data_fao[data_fao_item_colname].unique() 
    value_name = "Value" if "Value" in data_fao.columns else "value"
    data_fao[value_name] = data_fao[value_name].astype(str)
    data_fao[value_name] = data_fao[value_name].str.replace(',', '')
    data_fao[value_name] = pd.to_numeric(data_fao[value_name], errors='coerce')

    # 筛选出国家
    country_species = data[data_country_colname].unique()

    data_only_state = data[data['county'].isna() & data['state'].notna()] # 取出州的
    # 仅保留县
    data.dropna(subset=['county','state'],inplace=True)
    # data = data[data['county'].notna() & data['state'].notna()] # 仅保留县

    for animal in animal_species:
        
        # 首先判断这个动物种类是否在data里面
        if animal in data.columns:
            # 首先转化列里面数值保证能进行四则运算
            r[animal] = r[animal].astype(str)
            r[animal] = r[animal].str.replace(',', '')  # remove commas
            r[animal] = pd.to_numeric(r[animal], errors='coerce').fillna(0)

            # 首先转化列里面数值保证能进行四则运算
            data[animal] = data[animal].astype(str)
            data[animal] = data[animal].str.replace(',', '')  # remove commas
            data[animal] = pd.to_numeric(data[animal], errors='coerce')

            for country in country_species:
                # 获取FAO数据
                fao_value = data_fao[(data_fao[data_fao_item_colname] == animal) & (data_fao['Area'] == country)][value_name]
                if not fao_value.empty:
                    if pd.to_numeric(fao_value.iloc[0], errors='coerce') >= 0:
                        fao_value = fao_value.iloc[0]

                        # 计算国家内各县的总量(注意不要加重复了)
                        country_total = data_only_state[data_only_state[data_country_colname] == country][animal].sum() # 总量应该从州取
                        if country_total==0:
                            # 如果总量等于0，则从县取
                            country_total = data[data[data_country_colname] == country][animal].sum() # 总量应该从县取

                        if abs(fao_value-country_total) > 0.2*max(fao_value,country_total):
                            # 标记一下这一行，然后跳过,区分是有数据还是没有数据导致的
                            count_error += 1
                            if country_total==0 and fao_value != 0:
                                # 用fao_value代替country_total,country_total为0的化，国家里面的县也会为零，那这些县按照历史比例分配下去
                                # 找出最近的年份，其中 'animal' 列的值的总和大于等于 0
                                for offset in range(1, max(Year - 1980, 2021 - Year) + 1):
                                    recent_year = Year - offset if Year - offset >= 1980 else Year + offset
                                    if recent_year <= 2021 and r[(r[data_country_colname]==country) & (r['year']==recent_year)][animal].sum() >= 0:
                                        data.loc[data[data_country_colname] == country, animal] = r[(r[data_country_colname]==country) & (r['year']==recent_year)][animal] * fao_value
                                        break
                                count_zero += 1
                            else:
                                data.loc[data[data_country_colname] == country, "标记"] += "种类："+animal+","+"FAO总量："+str(fao_value)+","+"国家总量："+str(country_total)+"; "

                            # print("国家为{}，种类为{}的fao总量为{}，对应国家数据总量{}".format(country,animal,fao_value,country_total))
                            
                        # 如果国家总量大于0，则根据占比分配FAO数据
                        if country_total > 0:
                            # 大于零的情况可能是州不为0 ，但是县为0，所有当县为0 ，则总量要用之前的比例进行分配
                            if data[data[data_country_colname] == country][animal].sum()==0:
                                # 找出最近的年份，其中 'animal' 列的值的总和大于等于 0
                                for offset in range(1, max(Year - 1980, 2021 - Year) + 1):
                                    recent_year = Year - offset if Year - offset >= 1980 else Year + offset
                                    if recent_year <= 2021 and r[(r[data_country_colname]==country) & (r['year']==recent_year)][animal].sum() >= 0:
                                        data.loc[data[data_country_colname] == country, animal] = r[(r[data_country_colname]==country) & (r['year']==recent_year)][animal]  * fao_value
                                        break
                                
                            else:
                                data.loc[data[data_country_colname] == country, animal] = (data[animal] / country_total) * fao_value
    print("缺失占比为{}".format(count_zero/count_error))
    return data


In [34]:
import pandas as pd
r = pd.read_csv("D:/中科院数据下载/eurostat/animal_geo_ok/比例县.csv")

# 进行校对
# 与FAO数据校对
year_begin = 1980
year_end = 2021
for y in range(year_begin,year_end+1):
    data = pd.read_csv("D:/中科院数据下载/eurostat/animal_geo_ok/"+str(y)+".csv")
    data_fao = pd.read_csv("D:/中科院数据下载/eurostat/animal_geo_ok/FAO数据/"+str(y)+".csv")
    data_ok = single_proofread_r(data,data_fao,r,data_country_colname="country_label",data_fao_item_colname="Item")

    data_ok.to_csv("动物_fao_ok/"+str(y)+".csv",index=False,encoding="utf-8-sig")

缺失占比为0.8571428571428571
缺失占比为0.8586956521739131
缺失占比为0.8478260869565217
缺失占比为0.8369565217391305
缺失占比为0.8260869565217391
缺失占比为0.8131868131868132
缺失占比为0.8241758241758241
缺失占比为0.7666666666666667
缺失占比为0.7444444444444445
缺失占比为0.7222222222222222
缺失占比为0.6781609195402298
缺失占比为0.6704545454545454
缺失占比为0.7297297297297297
缺失占比为0.7222222222222222
缺失占比为0.7222222222222222
缺失占比为0.6410256410256411
缺失占比为0.625
缺失占比为0.6016949152542372
缺失占比为0.5714285714285714
缺失占比为0.5727272727272728
缺失占比为0.5042016806722689
缺失占比为0.45614035087719296
缺失占比为0.44545454545454544
缺失占比为0.4537037037037037
缺失占比为0.5045045045045045
缺失占比为0.656
缺失占比为0.7388059701492538
缺失占比为0.330188679245283
缺失占比为0.39473684210526316
缺失占比为0.3925233644859813
缺失占比为0.30275229357798167
缺失占比为0.2897196261682243
缺失占比为0.3425925925925926
缺失占比为0.3611111111111111
缺失占比为0.37962962962962965
缺失占比为0.36538461538461536
缺失占比为0.3867924528301887
缺失占比为0.38317757009345793
缺失占比为0.3434343434343434
缺失占比为0.3191489361702128
缺失占比为0.32608695652173914
缺失占比为0.23809523809523808


In [4]:
import pandas as pd 
# ARA匹配到动物
ARA = 'Main area (1000 ha)_Arable land'
animal = pd.read_csv("D:/中科院数据下载/eurostat/动物_fao_ok/1980（欧盟动物已校对）.csv")
crops = pd.read_csv("D:/中科院数据下载/eurostat/农作物_ok/1980.csv")

data = animal.merge(crops[[ARA,'county']],on='county',how='left')
data.to_csv("D:/中科院数据下载/eurostat/动物_fao_ok/1980（欧盟动物已校对）.csv")

In [5]:
import pandas as pd 
# ARA匹配到动物
ARA = 'Main area (1000 ha)_Arable land'
animal = pd.read_csv("D:/中科院数据下载/eurostat/动物_fao_ok/2021(欧盟动物以校对）.csv")
crops = pd.read_csv("D:/中科院数据下载/eurostat/农作物_ok/2021.csv")

data = animal.merge(crops[[ARA,'county']],on='county',how='left')
data.to_csv("D:/中科院数据下载/eurostat/动物_fao_ok/2021(欧盟动物以校对）.csv")